<a href="https://colab.research.google.com/github/clin0603/csci3052u-group-project/blob/main/milestone2_EDA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#imports

import os
import numpy as np
import pandas as pd


import matplotlib.pyplot as plt
import seaborn as seaborn_sns

## Research Question

## Data Source and License

In [ ]:
#TEMPLATE
# 1. Data Source & License declaration
DATA_SOURCE = "Consumer Fiancial Protection Bureau - HMDA Historial Data"
DATA_URL = "https://www.consumerfinance.gov/data-research/hmda/historic-data/?geo=nationwide&records=all-records&field_descriptions=labels"
LICENSE = "Publicly available"

YEARS = [2007, 2008, 2009]
GEOGRAPHY = "Nevada"
RECORDS = "All records"

## Data Card

In [ ]:
#TEMPLATE
file_paths = [
    "Cut_Mortage_Data07.csv",
    "Cut_Mortage_Data08.csv",
    "Cut_Mortage_Data09.csv"
]

if all(os.path.exists(file) for file in file_paths):
    # Load and combine the datasets
    df = pd.concat(
        [pd.read_csv(file, low_memory=False) for file in file_paths],
        ignore_index=True
    )
    print("=== DATA CARD ===")
    print(f"Source: {DATA_SOURCE}")
    print(f"License: {LICENSE}")
    print(f"Total Rows (Observations): {df.shape[0]:,}")
    print(f"Total Columns (Features): {df.shape[1]}")
else:
    print("Warning: One or more files not found. Please check your file paths.")

=== DATA CARD ===
Source: Consumer Fiancial Protection Bureau - HMDA Historial Data
License: Publicly available
Total Rows (Observations): 101,864
Total Columns (Features): 78


## Sample Inputs and Outputs

In [ ]:
#template

sample_features = ['as_of_year', 'loan_amount_000s', 'applicant_income_000s', 'loan_type', 'loan_purpose']
target_column = 'approved'

if set(sample_features).issubset(df.columns) and 'action_taken' in df.columns:
     # Keep only approved and denied applications
    df_model = df[df['action_taken'].isin([1, 2, 3])].copy()

    # Create binary target: 1 = Approved, 0 = Denied
    df_model[target_column] = df_model['action_taken'].isin([1, 2]).astype(int)

    # Select 2 approved and 2 denied applications
    sample = pd.concat([
        df_model[df_model[target_column] == 1].head(2),
        df_model[df_model[target_column] == 0].head(2)
    ])

    print("=== SAMPLE INPUTS (X) & OUTPUTS (y) ===")

    print("\nSample Input Rows (Features - X):")
    print(sample[sample_features])

    print("\nSample Outputs (Target - y):")
    print(sample[target_column])

else:
    print("Note: Check your sample features and target column.")
    display(df.head(3))

# Separate input features and target
X = df_model[sample_features]
y = df_model[target_column]

print("Input shape (X):", X.shape)
print("Output shape (y):", y.shape)

# Check the possible target values
print("Target values:", y.unique())

# Count approved and denied applications
print("\nTarget distribution:")
print(y.value_counts())



=== SAMPLE INPUTS (X) & OUTPUTS (y) ===

Sample Input Rows (Features - X):
    as_of_year  loan_amount_000s  applicant_income_000s  loan_type  \
0         2007                41                   84.0          1   
1         2007               295                  119.0          3   
7         2007               225                   53.0          1   
15        2007               285                   75.0          1   

    loan_purpose  
0              1  
1              3  
7              3  
15             1  

Sample Outputs (Target - y):
0     1
1     1
7     0
15    0
Name: approved, dtype: int64
Input shape (X): (74103, 5)
Output shape (y): (74103,)
Target values: [1 0]

Target distribution:
approved
1    54767
0    19336
Name: count, dtype: int64


The code above shows sample inputs and outputs from the mortgage datasets to help us understand what information will be used to predict loan approval. This sample only displays applications from 2007, the full dataset includes 2007, 2008, and 2009. The inputs include loan amount, applicant income, loan type, loan purpose, and year. The output shows whether each application was approved (1) or denied (0).

 After filtering the data, we have a total of 74,103 mortgage applications with 5 input features. Out of these applications, 54,767 (73.9%) were approved, while 19,336 (26.1%) were denied. This shows that our dataset is imbalanced, since there are significantly more approved applications than denied ones.

## Target and ML Task

We use **1 for approved** and **0 for denied**. Loans that were approved or given out are counted as approved, while rejected applications are counted as denied. Other outcomes, such as withdrawn or incomplete applications, are not included.

To make these predictions, we look at information such as the year, loan amount, applicant income, loan type, and loan purpose. By comparing data from **2007, 2008, and 2009**, we want to see how mortgage approvals and denials changed around the 2008 Financial Crisis and how accurately our model can predict these decisions.

## Initial EDA

In [ ]:
#TEMPLATE
# 1. Inspect the structure and dimensions
df.info()  # Rows, columns, memory usage, and data types
df.shape  # (number of rows, number of columns)

# 2. Preview the rows
df.head()  # View first 5 rows
df.tail()  # View last 5 rows
df.sample(5)  # View 5 random rows

# 3. Check data quality
df.isna().sum()  # Total missing (NaN) values per column
df.duplicated().sum()  # Total identical duplicate rows

# 4. Statistical summaries
df.describe()  # Summary statistics for numerical columns
df.describe(include=["object", "str"])  # Summary statistics for text/categorical columns

# 5. Column-specific checks
df["action_taken_name"].value_counts(dropna=False)
df["loan_purpose_name"].value_counts(dropna=False)

# 6. Actionable Missing Data Check (Only columns with missing values)
null_pct = (df.isna().sum() / len(df)) * 100
missing_report = null_pct[null_pct > 0].sort_values(ascending=False)
print("--- Columns with Missing Values (%) ---")
print(missing_report.head(20))  # Top 20 worst columns

# 7. Target Breakdown & Class Imbalance (Normalized %)
print("\n--- Target Distribution (Percentages) ---")
print(df["action_taken_name"].value_counts(normalize=True) * 100)

# 8. Relationship: Feature vs. Target (Cross-tabulation)
# See how approval/denial distributes across loan purposes
print("\n--- Loan Purpose vs. Action Taken ---")
pd.crosstab(df["loan_purpose_name"], df["action_taken_name"], normalize="index") * 100

<class 'pandas.DataFrame'>
RangeIndex: 101864 entries, 0 to 101863
Data columns (total 78 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   as_of_year                      101864 non-null  int64  
 1   respondent_id                   101864 non-null  str    
 2   agency_name                     101864 non-null  str    
 3   agency_abbr                     101864 non-null  str    
 4   agency_code                     101864 non-null  int64  
 5   loan_type_name                  101864 non-null  str    
 6   loan_type                       101864 non-null  int64  
 7   property_type_name              101864 non-null  str    
 8   property_type                   101864 non-null  int64  
 9   loan_purpose_name               101864 non-null  str    
 10  loan_purpose                    101864 non-null  int64  
 11  owner_occupancy_name            101864 non-null  str    
 12  owner_occupancy            

action_taken_name,Application approved but not accepted,Application denied by financial institution,Application withdrawn by applicant,File closed for incompleteness,Loan originated,Loan purchased by the institution,Preapproval request approved but not accepted,Preapproval request denied by financial institution
loan_purpose_name,,,,,,,,
Home improvement,6.339691,41.162579,9.809264,2.779292,33.678474,6.230699,0.000000,0.00000
Home purchase,4.304904,11.041742,4.845835,1.124684,56.648936,21.916697,0.036062,0.08114
Refinancing,4.427689,23.409821,9.844011,2.463888,44.657729,15.196861,0.000000,0.00000


## Risks and Limitations

Historical Data Quality:
Algorithm Explanation:
Data Leakage in Temporal (Time-Series) Data: (basically if we use future data to measure past data)

## Train/Validation/Test Strategy

In [ ]:
#template
#might have to fix trainign data sizes etc, AGAIN THIS IS A TEMPLATE just to try to save time

# Assuming you have a 'year' or 'quarter' column
time_col = 'year' # Change to your time column if available

if time_col in df.columns:
    print("=== TRAIN/VAL/TEST STRATEGY (Time-Based Split) ===")

    # Example logic: Train on pre-crisis, Val on early crisis, Test on peak crisis
    train_df = df[df[time_col] <= 2007]
    val_df = df[(df[time_col] == 2008)]
    test_df = df[df[time_col] >= 2009]

    print(f"Train Set (Pre-2008): {len(train_df)} rows")
    print(f"Validation Set (2008): {len(val_df)} rows")
    print(f"Test Set (2009+): {len(test_df)} rows")
else:
    print("=== TRAIN/VAL/TEST STRATEGY (Standard Train/Test Split) ===")
    from sklearn.model_selection import train_test_split

    # Fallback standard split if no explicit time column is set up yet
    # train/val/test standard split
    print("No time column found. Using standard random split code snippet.")

## Primary Evaluation Metric

In [ ]:
#Tempalte
#this is for logistic regression and not KNN, again just wanna put stuff incase it can be used later? apparently knn is bad for ho wmany columns we might get? so maybe we say how knn fails and then switch to this

from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

print("=== KNN FEASIBILITY & COMPUTE TEST ===")
try:
    # 1. Create a mock dataset with wildly different scales and a missing value (NaN)
    # Feature 1: Assets in billions (e.g., 500,000 to 2,000,000)
    # Feature 2: Capital ratio (e.g., 0.05 to 0.15)
    X_dummy = np.array([[1000000, 0.08], [np.nan, 0.12], [2000000, 0.05], [500000, 0.15]])
    y_dummy = np.array([0, 0, 1, 1])

    # 2. Impute missing values first
    imputer = SimpleImputer(strategy='mean')
    X_imputed = imputer.fit_transform(X_dummy)

    # 3. Scale features (CRITICAL for KNN so large numbers don't overpower small ones)
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_imputed)

    # 4. Train KNN model
    knn = KNeighborsClassifier(n_neighbors=3)
    knn.fit(X_scaled, y_dummy)
    preds = knn.predict(X_scaled)

    print("Compute plan status: PASSED! (Imputer, StandardScaler, and KNeighborsClassifier executed successfully).")
except Exception as e:
    print(f"Compute check failed: {e}")

## Feasibility and Compute Plan